<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_02_2_pytorch_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# T81-558: Applications of Deep Neural Networks

**Module 2: PyTorch for Neural Networks**

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 2 Material

* Part 2.1: Numeric Processing with PyTorch [[Video]]() [[Notebook]](t81_558_class_02_1_pytorch_numerical.ipynb)
* **Part 2.2: Deep Learning with PyTorch** [[Video]]() [[Notebook]](t81_558_class_02_2_pytorch_neural.ipynb)
* Part 2.3: Preprocessing for PyTorch [[Video]]() [[Notebook]](t81_558_class_02_3_feature_encode.ipynb)
* Part 2.4: nn.Module vs nn.Sequential for PyTorch [[Video]]() [[Notebook]](t81_558_class_02_4_pytorch_class_sequence.ipynb)
* Part 2.5: Beyond the CPU[[Video]]() [[Notebook]](t81_558_class_02_5_beyond_cpu.ipynb)

# Google Colab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.


In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 3.2)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: using Google CoLab
Using device: cuda


# Part 2.2: Deep Learning with PyTorch



We now move from basic tensor operations to applying PyTorch to simple machine learning tasks. In this section, we will explore regression and classification problems, which form the foundation of most practical AI applications. Regression focuses on predicting continuous values, while classification assigns inputs to discrete categories. These examples introduce the core workflow in PyTorch, including defining models, computing loss, and training with optimization.

## Simple PyTorch Regression: MPG

This example shows how to encode the MPG dataset for regression and predict values. We will see if we can predict a car's miles per gallon (MPG) based on its weight, number of cylinders, engine size, and other features. We will begin by reading the MPG dataset.


In [2]:
import pandas as pd
import torch
import numpy as np
import torch.nn as nn
from sklearn.metrics import mean_squared_error

# Load the Auto MPG dataset, treating "NA" and "?" as missing values
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv",
    na_values=["NA", "?"]
)

# Preserve the car names before dropping columns
cars = df["name"]

# Impute missing horsepower values with the column median
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Feature columns used as model inputs
feature_cols = ["cylinders", "displacement", "horsepower",
                "weight", "acceleration", "year", "origin"]

# Create input tensor x and target tensor y (float32 for PyTorch)
x = torch.tensor(df[feature_cols].values, dtype=torch.float32)
y = torch.tensor(df["mpg"].values, dtype=torch.float32).unsqueeze(1)  # shape (n, 1)

print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")

x shape: torch.Size([398, 7])
y shape: torch.Size([398, 1])


We use Pandas to load the CSV file, as previously demonstrated. We will save the car names, though they do not help predict MPG. Horsepower has missing values, so we substitute the median for any missing values. Next, we convert Pandas to NumPy, and NumPy to PyTorch. We select only the fields we wish to use for prediction. As previously discussed, we designed the Net class to detect the size of this data and add the appropriate count of input neurons.

You define your neural network in the **Sequence** above. We will see later that you may also construct a neural network using a Python class. In this case, we have a neural network with an input layer equal to the number of inputs you specify from the MPG dataset. The neural network connects these inputs to 50 neurons in the first hidden layer, which are connected to 25 neurons in the second layer. The output neuron count for a layer must always match the input count of the next layer.

For this book, we will generally always use the ReLU activation function for hidden layers. The output layer of a regression neural network will use no transfer function, as in this MPG example. For classification, we use logistic regression for binary classification (just two classes) or log-softmax for two or more classes.

For the neural network to perform correctly, everything must align. The **sequence** must specify all layers with the same number of outputs as inputs for each connection.

We are ready to create the neural network, loss function, and optimizer class with the data loaded.

In [3]:
print(f"Using device: {device}")

# Move tensors to the selected device
x = x.to(device)
y = y.to(device)

# Define the network: 7 inputs -> Linear(50) -> ReLU -> Linear(25) -> ReLU -> Linear(1)
model = nn.Sequential(
    nn.Linear(x.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
).to(device)

# Mean Squared Error loss for regression
loss_fn = nn.MSELoss()

# Adam optimizer with a learning rate of 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(model)

Using device: cuda
Sequential(
  (0): Linear(in_features=7, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=25, bias=True)
  (3): ReLU()
  (4): Linear(in_features=25, out_features=1, bias=True)
)


We create the neural network with one input equal to the number of columns in the x-input data. We specify a single output neuron that predicts MPG. Next, we define MSELoss as the error function, which is a common choice for regression. We will use the Adam optimizer with a learning rate of 0.01 to train the network. Adam is a common choice, and 0.01 is a good starting point for the learning rate. The learning rate should never be above 1.0. A learning rate that is too large will fail to learn the problem thoroughly, and one that is too low will take a long time to train. We will see more advanced methods for choosing the learning rate, including schedules that change it throughout training.

In [4]:
EPOCHS = 1000
LOG_INTERVAL = 100

model.train()
for epoch in range(1, EPOCHS + 1):
    # Forward pass
    pred = model(x)
    loss = loss_fn(pred, y)

    # Backward pass and weight update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Report progress every LOG_INTERVAL epochs
    if epoch % LOG_INTERVAL == 0:
        print(f"Epoch {epoch:>4d}/{EPOCHS}  |  MSE Loss: {loss.item():.4f}")

Epoch  100/1000  |  MSE Loss: 153.5025
Epoch  200/1000  |  MSE Loss: 117.2576
Epoch  300/1000  |  MSE Loss: 80.2307
Epoch  400/1000  |  MSE Loss: 54.1049
Epoch  500/1000  |  MSE Loss: 41.9226
Epoch  600/1000  |  MSE Loss: 33.5115
Epoch  700/1000  |  MSE Loss: 27.1579
Epoch  800/1000  |  MSE Loss: 22.0292
Epoch  900/1000  |  MSE Loss: 17.9454
Epoch 1000/1000  |  MSE Loss: 15.0033


We now loop over 1,000 epochs and train the neural network; an epoch is one complete pass over the training set. We zero the gradients, so training from the previous epoch does not influence the current epoch. We present the entire training set to the model as one large batch. Later, we will see more advanced methods for segmenting the data. We apply the loss function and use backpropagation to compute gradients for updating the neural network weights.

## Introduction to Neural Network Hyperparameters

This network includes several hidden layers with 50 and 25 neurons, respectively. You might be wondering how the programmer chose these numbers. Selecting a hidden neuron structure is one of the most common questions about neural networks. Unfortunately, there is no right answer. These are hyperparameters. They are settings that can affect neural network performance, yet there are no clearly defined ways to set them.

In general, more hidden neurons mean more capability to fit complex problems. However, too many neurons can lead to overfitting and lengthy training times. Too few can lead to underfitting the problem and result in lower accuracy. Also, the number of layers you have is another hyperparameter. In general, more layers allow the neural network to perform more of its feature engineering and data preprocessing. But this also comes at the expense of training times and the risk of overfitting. In general, you will see that neuron counts start larger near the input layer and tend to shrink towards the output layer in a triangular fashion.

Some techniques use machine learning to optimize these values. These will be discussed in [Module 8.3](t81_558_class_08_3_pytorch_hyperparameters.ipynb).

## Regression Prediction

Next, we will perform actual predictions. The program assigns these predictions to the **pred** variable. These are all MPG predictions from the neural network. Notice that this is a 2D array? You can always see the dimensions of what PyTorch returns by printing out **pred.shape**. Neural networks can return multiple values, so the result is always an array. Here, the neural network only returns one value per prediction (there are 398 cars, so 398 predictions). However, a 2D range is needed because the neural network may return more than one value.


With the objects created, we can now train the neural network.


In [5]:
model.eval()
with torch.no_grad():
    # Generate predictions for the full input tensor
    pred = model(x)

print(f"Shape: {pred.shape}")
print("\nFirst 10 predictions:")
print(pred[:10])

Shape: torch.Size([398, 1])

First 10 predictions:
tensor([[14.6432],
        [12.9290],
        [14.2850],
        [15.1121],
        [14.7038],
        [ 8.8835],
        [ 7.8608],
        [ 8.3250],
        [ 8.0279],
        [10.6760]], device='cuda:0')


We would like to see how good these predictions are. We know the correct MPG for each car, so we can measure how close the neural network was. We will first see how we calculate RMSE with standard Scikit-learn metrics. To utilize Sklearn, we must bring the predictions back to the CPU and detach them from the neural network graph. The following code accomplishes this with **cpu().detach()**.


In [6]:
from sklearn.metrics import root_mean_squared_error

# Move tensors to CPU and convert to NumPy for scikit-learn compatibility
y_np    = y.cpu().numpy()
pred_np = pred.cpu().numpy()

rmse_sklearn = root_mean_squared_error(y_np, pred_np)
print(f"RMSE (scikit-learn): {rmse_sklearn:.4f}")

RMSE (scikit-learn): 3.8703


We can accomplish the same task entirely within PyTorch with less code. It is important to know how to perform these calculations both with PyTorch and Scikit-learn.


In [7]:
# Compute RMSE entirely in PyTorch (no NumPy conversion needed)
rmse_torch = torch.sqrt(loss_fn(pred, y))
print(f"RMSE (PyTorch): {rmse_torch.item():.4f}")

RMSE (PyTorch): 3.8703


The number printed above is the average number of predictions above or below the expected output. We can also print the first 10 cars with predicted and actual MPG.


In [8]:
# Display the first 10 predictions alongside actual MPG and car name
print("First 10 predictions:\n")
for i in range(10):
    car_name      = cars.iloc[i]
    actual_mpg    = y.cpu().numpy()[i][0]
    predicted_mpg = pred.cpu().numpy()[i][0]
    print(f"{i}. Car name: {car_name}, MPG: {actual_mpg:.1f}, predicted MPG: {predicted_mpg:.2f}")

First 10 predictions:

0. Car name: chevrolet chevelle malibu, MPG: 18.0, predicted MPG: 14.64
1. Car name: buick skylark 320, MPG: 15.0, predicted MPG: 12.93
2. Car name: plymouth satellite, MPG: 18.0, predicted MPG: 14.29
3. Car name: amc rebel sst, MPG: 16.0, predicted MPG: 15.11
4. Car name: ford torino, MPG: 17.0, predicted MPG: 14.70
5. Car name: ford galaxie 500, MPG: 15.0, predicted MPG: 8.88
6. Car name: chevrolet impala, MPG: 14.0, predicted MPG: 7.86
7. Car name: plymouth fury iii, MPG: 14.0, predicted MPG: 8.32
8. Car name: pontiac catalina, MPG: 14.0, predicted MPG: 8.03
9. Car name: amc ambassador dpl, MPG: 15.0, predicted MPG: 10.68


## Simple TensorFlow Classification: Iris

Classification is how a neural network attempts to assign an input to one or more classes. The simplest way to evaluate a classification network is to track the percentage of training set items misclassified. We typically score human results in this manner. For example, you might have taken multiple-choice exams in school in which you had to shade in a bubble for choices A, B, C, or D. If you chose the wrong letter on a 10-question exam, you would earn a 90%. In the same way, we can grade computers; however, most classification algorithms do not merely assign A, B, C, or D. Computers typically report a classification along with their confidence in each class. Figure 2.EXAM shows how a computer and a human might respond to question number 1 on an exam.

**Figure 2.EXAM: Classification Neural Network Output**
![Classification Neural Network Output](https://data.heatonresearch.com/images/wustl/class/class-multi-choice.png "Classification Neural Network Output")

As you can see, the human test taker marked the first question as "B." However, the computer test taker had an 80% (0.8) confidence in "B" and was also somewhat sure with 10% (0.1) on "A." The computer then distributed the remaining points to the other two. In the simplest sense, the machine would get 80% of the score for this question if the correct answer were "B." The computer would get only 5% (0.05) of the points if the correct answer were "D."

We previously saw how to train a neural network to predict a car's MPG. Based on four measurements, we will now see how to predict a class, such as the type of iris flower. The code to classify iris flowers is similar to MPG; however, there are several important differences:

- The output neuron count matches the number of classes (in the case of Iris, 3).
- The output layer produces raw logits (no activation function). **CrossEntropyLoss** applies softmax internally, so no explicit activation is needed on the output.
- The loss function is **CrossEntropyLoss**.
- We call the **train** function to inform PyTorch that we are now in training mode.
- Later, we call the **eval** function to inform PyTorch that we are done training and are evaluating the network.

Softmax is commonly used in classification tasks because it converts a model's output into a probability distribution over multiple classes. By applying the softmax function to a model's logits (raw outputs), we obtain normalized probabilities that sum up to one, indicating the likelihood of each class. This technique lets us interpret the model's predictions as confidence scores for different classes and make decisions based on the highest probability.

Log softmax, on the other hand, offers several advantages over regular softmax. One advantage is that it helps alleviate numerical instability issues that can occur when dealing with both small and large values in softmax computations. By taking the logarithm of the softmax probabilities, we work in the logarithmic domain, which can improve numerical stability and prevent overflow or underflow errors. The log-softmax is also useful when using the negative log-likelihood loss function, as it simplifies calculations by eliminating the need to compute the actual probabilities. This technique is particularly beneficial for neural network training, as it simplifies optimization. Log softmax provides a practical and efficient way to handle classification problems, offering numerical stability and simplifying loss computation.

In PyTorch, rather than manually adding a LogSoftmax layer, we use nn.CrossEntropyLoss, which combines the log-softmax and negative log-likelihood loss in a single numerically stable operation. The output layer therefore produces raw logits.


In [9]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder

# Load the Iris dataset
df = pd.read_csv(
    "https://data.heatonresearch.com/data/t81-558/iris.csv",
    na_values=["NA", "?"]
)

# Build input tensor from the four measurement columns
x = torch.tensor(
    df[["sepal_l", "sepal_w", "petal_l", "petal_w"]].values,
    dtype=torch.float32
)

# Encode the target species strings as integer class indices
le = LabelEncoder()
y = torch.tensor(le.fit_transform(df["species"].values), dtype=torch.long)

# Store the human-readable class names for later decoding
species = le.classes_

# Define the sequential classification model
model = nn.Sequential(
    nn.Linear(x.shape[1], 50),   # input layer -> 50 hidden units
    nn.ReLU(),
    nn.Linear(50, 25),            # 50 -> 25 hidden units
    nn.ReLU(),
    nn.Linear(25, len(species))   # 25 -> one logit per class
)

# Cross-entropy loss is standard for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimizer with a moderate learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(1, 1001):
    optimizer.zero_grad()           # clear gradients from previous step
    output = model(x)               # forward pass
    loss = criterion(output, y)     # compute loss
    loss.backward()                 # backpropagate
    optimizer.step()                # update weights

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, loss: {loss.item():.4f}")

Epoch 100, loss: 0.0507
Epoch 200, loss: 0.0453
Epoch 300, loss: 0.0422
Epoch 400, loss: 0.0404
Epoch 500, loss: 0.0400
Epoch 600, loss: 0.0405
Epoch 700, loss: 0.0429
Epoch 800, loss: 0.0396
Epoch 900, loss: 0.0392
Epoch 1000, loss: 0.0389


In [10]:
# Display the unique species labels learned by the encoder
print("Species classes:", species)

Species classes: ['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']


Now that you have a trained neural network, we would like to use it. The following code uses our neural network. Exactly like before, we will generate predictions. Notice that three values come back for each of the 150 iris flowers. There were three types of iris (Iris-setosa, Iris-versicolor, and Iris-virginica). We call the **no_grad** function to inform PyTorch that we are no longer training and wish to evaluate.


In [11]:
# Run inference over the entire dataset (no gradient tracking needed)
with torch.no_grad():
    pred = model(x)

# pred contains raw logits — one row per sample, one column per class
print("Prediction tensor shape:", pred.shape)
print("\nFirst 10 raw predictions (logits):")
print(pred[:10])

Prediction tensor shape: torch.Size([150, 3])

First 10 raw predictions (logits):
tensor([[ 29.3565,  17.7208, -39.3617],
        [ 27.5617,  16.7112, -37.0195],
        [ 28.2090,  17.0029, -37.7833],
        [ 27.1119,  16.4381, -36.4174],
        [ 29.5169,  17.7901, -39.5493],
        [ 27.8878,  17.6591, -37.9274],
        [ 27.6317,  16.6881, -37.0521],
        [ 28.5859,  17.3090, -38.3843],
        [ 26.5503,  16.0736, -35.6284],
        [ 28.2508,  17.1173, -37.9376]])


Usually, the program considers the column with the highest neural network prediction to be the neural network's prediction. It is easy to convert the predictions to the expected iris species. The argmax function finds the index of the maximum prediction for each row.


In [12]:
# The predicted class is the column with the highest logit in each row
predicted_indices = torch.argmax(pred, dim=1)
print("Predicted category indices:")
print(predicted_indices)

print("\nExpected category indices:")
print(y)

Predicted category indices:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2])

Expected category indices:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

Of course, it is straightforward to turn these indices back into iris species. We use the species list that we created earlier.


In [13]:
# Map predicted integer indices back to species name strings
predicted_species = [species[i] for i in predicted_indices]
print("Predicted species names:")
print(predicted_species)

Predicted species names:
['Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-setosa', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolor', 'Iris-versicolo

Accuracy might be a more easily understood error metric. It is essentially a test score. For all of the iris predictions, what percent were correct? The downside is that it does not account for how confident the neural network was in each prediction.


In [14]:
# Accuracy = fraction of samples where prediction matches the true label
correct = (predicted_indices == y).sum().item()
total = len(y)
accuracy = correct / total
print(f"Accuracy: {correct}/{total} = {accuracy:.4f} ({accuracy * 100:.2f}%)")

Accuracy: 148/150 = 0.9867 (98.67%)


The code below performs two ad hoc predictions. The first prediction is a single iris flower, and the second predicts two iris flowers. Notice that the **argmax** in the second prediction requires **axis=1**? Since we have a 2D array now, we must specify which axis to take the **argmax** over. The value **axis=1** specifies we want the max column index for each row.


In [15]:
# Single flower sample: sepal_l, sepal_w, petal_l, petal_w
single_flower = torch.tensor([[5.0, 3.0, 4.0, 2.0]], dtype=torch.float32)

with torch.no_grad():
    single_pred = model(single_flower)

single_class = species[torch.argmax(single_pred, dim=1).item()]
print("Raw prediction (logits):", single_pred)
print("Predicted species:", single_class)

Raw prediction (logits): tensor([[-11.2539,   8.5260,   7.7640]])
Predicted species: Iris-versicolor


You can also predict two sample flowers.


In [16]:
# Two flower samples evaluated together in one forward pass
two_flowers = torch.tensor(
    [[5.0, 3.0, 4.0, 2.0],
     [5.2, 3.5, 1.5, 0.8]],
    dtype=torch.float32
)

with torch.no_grad():
    two_preds = model(two_flowers)

two_classes = [species[i] for i in torch.argmax(two_preds, dim=1)]
print("Raw predictions (logits):")
print(two_preds)
print("\nPredicted species:", two_classes)

Raw predictions (logits):
tensor([[-11.2539,   8.5260,   7.7640],
        [ 23.2090,  15.7805, -32.0947]])

Predicted species: ['Iris-versicolor', 'Iris-setosa']
